# Eco-asistente: análisis y preparación del dataset

Este notebook carga el CSV adjunto `dataset_eco_asistente.csv`, explora las categorías, prepara los datos y entrena y evalúa un clasificador. Funciona en Google Colab y en Jupyter.

**Dataset de elaboración propia para fines educativos:** el CSV actual contiene 689 filas y 623 descripciones distintas. Las categorías están desequilibradas, así que revisa el gráfico de distribución antes de interpretar las métricas. El dataset se ha ampliado desde una versión inicial; no son observaciones recogidas de usuarios. Usamos como esquema de referencia las cinco fracciones habituales en España: papel y cartón (azul), vidrio (verde), envases ligeros de plástico/metal/brik (amarillo), biorresiduos (marrón) y resto. **No existe un único modelo de contenedores idéntico en todos los municipios**: MITECO describe distintos modelos de separación y los servicios locales pueden tener instrucciones específicas. Por eso, el proyecto ofrece una orientación general para España, no sustituye las indicaciones municipales. Los residuos especiales, voluminosos o peligrosos quedan fuera del alcance inicial.

Referencia principal: [MITECO: modelos de gestión y fracciones municipales](https://www.miteco.gob.es/es/calidad-y-evaluacion-ambiental/temas/prevencion-y-gestion-residuos/flujos/domesticos/gestion/modelo_gestion.html) · [MITECO: recogida de envases](https://www.miteco.gob.es/gl/calidad-y-evaluacion-ambiental/temas/prevencion-y-gestion-residuos/flujos/envases/como-se-recogen.html).

**Importante:** son ejemplos preparados manualmente, no observaciones recogidas de usuarios. No representan todos los residuos posibles. La tabla de categorías es un punto de partida que debes revisar.

## 1. Cargar el archivo CSV

Adjunta `dataset_eco_asistente.csv` a Colab con el botón de archivos, o ejecútalo y selecciónalo cuando aparezca el selector. En Jupyter, deja el CSV en la carpeta `datos` o junto al notebook.

In [ ]:
from pathlib import Path
import pandas as pd

# Busca primero las ubicaciones habituales al ejecutar desde la carpeta del proyecto.
candidatos = [
    Path("datos/dataset_eco_asistente.csv"),
    Path("../datos/dataset_eco_asistente.csv"),
    Path("dataset_eco_asistente.csv"),
]
ruta_csv = next((p for p in candidatos if p.exists()), None)

if ruta_csv is None:
    try:
        from google.colab import files
        subidos = files.upload()  # Elige el CSV adjunto desde tu ordenador
        ruta_csv = Path(next(iter(subidos)))
    except ImportError:
        import tkinter as tk
        from tkinter.filedialog import askopenfilename
        ventana = tk.Tk(); ventana.withdraw()
        seleccion = askopenfilename(title="Selecciona dataset_eco_asistente.csv", filetypes=[("CSV", "*.csv")])
        if not seleccion:
            raise FileNotFoundError("No se seleccionó el CSV. Colócalo junto al notebook o en la carpeta datos.")
        ruta_csv = Path(seleccion)

df = pd.read_csv(ruta_csv)
print(f"Archivo cargado: {ruta_csv}")
df.head()

## 2. Revisar estructura y calidad

Comprobamos columnas, tipos, valores vacíos y filas repetidas antes de analizar las etiquetas.

In [ ]:
print("Filas y columnas:", df.shape)
display(df.head(10))
df.info()
print("\nValores vacíos por columna:\n", df.isna().sum())
print("\nFilas duplicadas exactas:", df.duplicated().sum())

## 3. Limpiar y validar

Normalizamos espacios y mayúsculas/minúsculas. Conservamos los acentos porque Scikit-learn puede procesar el texto y así las frases siguen siendo legibles. Quitamos filas vacías y duplicados exactos. Después verificamos que solo existan las cinco categorías acordadas.

In [ ]:
categorias_esperadas = {"amarillo", "azul", "gris", "marron", "punto_limpio", "verde"}

df = df[["residuo", "categoria"]].copy()
for columna in ["residuo", "categoria"]:
    df[columna] = df[columna].astype("string").str.strip().str.lower()
df = df.dropna(subset=["residuo", "categoria"])
df = df[(df["residuo"] != "") & (df["categoria"] != "")]
df = df.drop_duplicates().reset_index(drop=True)

print("Categorías encontradas:", sorted(df["categoria"].unique()))
inesperadas = set(df["categoria"].unique()) - categorias_esperadas
print("Etiquetas inesperadas:", inesperadas)
assert not inesperadas, "Revisa las etiquetas antes de continuar."
df.head()

## 4. Analizar las categorías

Contamos los ejemplos por categoría y representamos el equilibrio. Si una categoría tiene muchos menos ejemplos, el clasificador puede aprenderla peor.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Guarda las gráficas en una carpeta para poder insertarlas en la presentación.
CARPETA_GRAFICOS = Path("graficos")
CARPETA_GRAFICOS.mkdir(exist_ok=True)

conteo = df["categoria"].value_counts().reindex(sorted(categorias_esperadas), fill_value=0)
display(conteo.rename("cantidad de ejemplos").to_frame())
ax = conteo.plot(kind="bar", color="#4c956c", title="Ejemplos por categoría")
ax.set_xlabel("Categoría"); ax.set_ylabel("Número de ejemplos"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.savefig(CARPETA_GRAFICOS / "01_ejemplos_por_categoria.png", dpi=200, bbox_inches="tight"); plt.show()

## 5. Revisar duplicados de texto y posibles conflictos

Un mismo texto no debería tener dos etiquetas distintas. Las frases casi idénticas también pueden hacer que la evaluación parezca mejor de lo que realmente es; conviene revisar los ejemplos y no crear variantes casi iguales solo para inflar el dataset.

In [ ]:
conflictos = df.groupby("residuo")["categoria"].nunique()
conflictos = conflictos[conflictos > 1]
print("Textos con etiquetas contradictorias:", len(conflictos))
if len(conflictos):
    display(df[df["residuo"].isin(conflictos.index)].sort_values("residuo"))
else:
    print("No se encontraron etiquetas contradictorias exactas.")

print("\nEjemplos por categoría:")
for categoria, grupo in df.groupby("categoria"):
    print(f"\n{categoria} ({len(grupo)}):")
    print(", ".join(grupo["residuo"].head(8)))

## 6. Preparar los datos para el modelo

`X` contiene las descripciones y `y` las categorías que queremos predecir. En las secciones siguientes dividimos los datos en entrenamiento y prueba y convertimos el texto a números con `CountVectorizer`. No eliminamos los acentos: el vectorizador de Scikit-learn prepara las palabras para el modelo.

In [ ]:
X = df["residuo"].astype(str)
y = df["categoria"].astype(str)

print("Ejemplos disponibles:", len(X))
print("Etiquetas:", sorted(y.unique()))
print("\nTexto de ejemplo:", X.iloc[0])
print("Etiqueta correspondiente:", y.iloc[0])

## 7. Entrenar y evaluar el clasificador

Reservamos una parte de los ejemplos para comprobar el modelo con textos que no usó durante el entrenamiento. `CountVectorizer` convierte las palabras en frecuencias y `LogisticRegression` aprende a asociarlas con las categorías. El `Pipeline` une ambos pasos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = make_pipeline(
    CountVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42)
)
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

print(f"Ejemplos para entrenar: {len(X_train)}")
print(f"Ejemplos reservados para evaluar: {len(X_test)}")
print(f"Exactitud: {accuracy_score(y_test, predicciones):.2%}")
print("\nMétricas por categoría:")
print(classification_report(y_test, predicciones, zero_division=0))

## 8. Ver exactamente qué acierta y qué confunde

La tabla de abajo permite inspeccionar cada ejemplo reservado. `acierto=True` significa que la predicción coincide con su etiqueta; las filas con `False` son los errores concretos que conviene comentar y quizá añadir al dataset con cuidado.

In [ ]:
resultado = pd.DataFrame({
    "residuo": X_test.to_numpy(),
    "categoria_real": y_test.to_numpy(),
    "categoria_predicha": predicciones,
})
resultado["acierto"] = resultado["categoria_real"] == resultado["categoria_predicha"]
display(resultado.sort_values(["acierto", "categoria_real", "residuo"]))

print("Aciertos:", int(resultado["acierto"].sum()), "de", len(resultado))
print("Errores:", int((~resultado["acierto"]).sum()), "de", len(resultado))
print("\nSolo los errores:")
errores = resultado.loc[~resultado["acierto"]]
if errores.empty:
    print("No hubo errores en esta partición. Eso no garantiza que el modelo acierte todos los casos nuevos.")
else:
    display(errores)

## 9. Gráfico de aciertos y errores por categoría

Este gráfico compara cuántos residuos de cada categoría clasificó bien y cuántos confundió en el conjunto de prueba. Como la prueba contiene pocos ejemplos, muestra también los recuentos y no solo porcentajes.


In [ ]:
resumen_eval = (
    resultado.assign(resultado_eval=resultado["acierto"].map({True: "Acierto", False: "Error"}))
    .groupby(["categoria_real", "resultado_eval"]).size()
    .unstack(fill_value=0)
    .reindex(orden if "orden" in globals() else sorted(y.unique()), fill_value=0)
)
resumen_eval = resumen_eval.reindex(columns=["Acierto", "Error"], fill_value=0)
ax = resumen_eval.plot(kind="bar", color={"Acierto": "#4c956c", "Error": "#e76f51"}, figsize=(9, 5))
ax.set_title("Evaluación de residuos por categoría (conjunto de prueba)")
ax.set_xlabel("Categoría real"); ax.set_ylabel("Número de residuos")
ax.legend(title="Resultado del modelo")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "02_aciertos_y_errores.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Matriz de confusión

Cada fila representa la categoría real y cada columna la categoría que predijo el modelo. Los números de la diagonal son aciertos; los valores fuera de la diagonal muestran qué categorías se confundieron.

In [ ]:
orden = sorted(y.unique())
ConfusionMatrixDisplay.from_predictions(
    y_test, predicciones, labels=orden, display_labels=orden,
    xticks_rotation=35, cmap="Greens", values_format="d"
)
plt.title("Matriz de confusión del conjunto de prueba")
plt.tight_layout()
plt.savefig(CARPETA_GRAFICOS / "03_matriz_de_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

### Cómo explicarlo en la defensa

> «Separamos los datos en entrenamiento y prueba. El modelo aprendió con los ejemplos de entrenamiento. Después comparamos sus predicciones sobre los ejemplos reservados con sus categorías reales. La tabla enseña los residuos concretos que acertó o confundió, y la matriz de confusión resume esos cruces por categoría».

Este dataset es pequeño y está elaborado para el proyecto; una partición de prueba también es pequeña. Por eso, comenta los ejemplos y errores observados junto con las métricas y presenta los resultados como una demostración inicial, no como garantía para cualquier residuo de España.

## Qué hemos aprendido y siguientes pasos

- El dataset tiene una descripción (`residuo`) y una etiqueta (`categoria`).
- Primero revisamos el equilibrio, los valores vacíos y las etiquetas inconsistentes.
- La tabla es pequeña y hecha a mano: sirve para aprender el flujo, pero puede generalizar mal a residuos no incluidos.
- El notebook entrena el clasificador, mide su rendimiento y muestra los residuos que confunde.
- La app debe explicar su alcance y evitar tratar la predicción como una regla universal de reciclaje.

## Descargar los gráficos para la presentación

Al ejecutar el notebook completo se guardan tres imágenes PNG en la carpeta `graficos`: distribución de ejemplos por categoría, aciertos y errores por categoría, y matriz de confusión. En Colab, abre el panel **Archivos** para descargar esas imágenes; en Jupyter, busca la carpeta `graficos` junto al notebook. Los gráficos se generan con la partición de prueba de esta ejecución.
